In [3]:
import pandas as pd
import numpy as np
import time
import ephem
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

# 1. Cargar y ordenar
df_hmm_ready = pd.read_csv('../data/processed/hmm.csv')
df_hmm_ready = df_hmm_ready.sort_values(['animal_id', 'trayectoria_id', 'date'])

# 2. Crear target por segmento y filtrar NaNs
df_hmm_ready['target_cell'] = df_hmm_ready.groupby('trayectoria_id')['cell_id'].shift(-1)
df_hmm_ready_filtered = df_hmm_ready.dropna(subset=['target_cell']).copy()

In [4]:
# --- NUEVA CELDA 2 (COMBINADA) ---

# 1. Preparar variables temporales básicas
df_hmm_ready['date'] = pd.to_datetime(df_hmm_ready['date'])
df_hmm_ready['mes_num'] = df_hmm_ready['date'].dt.month
df_hmm_ready['dia_semana'] = df_hmm_ready['date'].dt.dayofweek

# 2. Calcular Fase Lunar con ephem
def get_lunar_phase(fecha):
    m = ephem.Moon()
    m.compute(fecha) 
    return m.phase / 100

print("Calculando fases lunares...")
df_hmm_ready['ciclo_lunar'] = df_hmm_ready['date'].apply(get_lunar_phase)

# 3. AHORA filtramos para crear el dataset de entrenamiento
# Esto asegura que df_hmm_ready_filtered ya traiga 'ciclo_lunar' y 'dia_semana'
df_hmm_ready_filtered = df_hmm_ready.dropna(subset=['target_cell']).copy()

# 4. Definir lista definitiva de features
features = [
    'grid_x', 'grid_y', 'step_length', 'turning_angle', 'bearing', 
    'estado_hmm', 'veg_low', 'veg_high', 'mes_num', 
    'dia_semana', 'ciclo_lunar'
]

X = df_hmm_ready_filtered[features]
y_all_raw = df_hmm_ready_filtered['target_cell']

print("¡Listo! Variables añadidas y dataset filtrado.")

Calculando fases lunares...
¡Listo! Variables añadidas y dataset filtrado.


In [5]:
# Sustituye tu bloque #4 por este:

X_train_list, X_test_list = [], []
y_train_list, y_test_list = [], []

# Agrupamos por animal para asegurar que cada uno tenga su 80% de entrenamiento y 20% de test
for animal, df_animal in df_hmm_ready_filtered.groupby('animal_id'):
    # Ordenar por fecha por si acaso
    df_animal = df_animal.sort_values('date')
    
    n = len(df_animal)
    if n < 5: continue # Omitir aves con poquísimos datos si las hubiera
    
    split_idx = int(n * 0.8)
    
    # Dividir datos del ave
    X_train_list.append(df_animal[features].iloc[:split_idx])
    X_test_list.append(df_animal[features].iloc[split_idx:])
    y_train_list.append(df_animal['target_cell'].iloc[:split_idx])
    y_test_list.append(df_animal['target_cell'].iloc[split_idx:])

# Concatenar todos los resultados
X_train_raw = pd.concat(X_train_list)
X_test_raw = pd.concat(X_test_list)
y_train_raw = pd.concat(y_train_list)
y_test_raw = pd.concat(y_test_list)

print(f"Dataset listo. Registros Train: {len(X_train_raw)}, Registros Test: {len(X_test_raw)}")

Dataset listo. Registros Train: 16323, Registros Test: 4137


In [6]:
# 5. LabelEncoder basado SOLO en entrenamiento
le_final = LabelEncoder()
y_train = le_final.fit_transform(y_train_raw)

# 6. Filtrar Test para que solo tenga celdas vistas en Train
mask_test = y_test_raw.isin(le_final.classes_)
X_test = X_test_raw[mask_test].copy() # .copy() para evitar warnings de SettingWithCopy
y_test = le_final.transform(y_test_raw[mask_test])

X_train = X_train_raw

In [7]:
# 7. Modelos
#Diccionario de modelos ACTUALIZADO para evitar el 0% de Accuracy
num_clases = len(le_final.classes_)

modelos = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        random_state=42, 
        objective='multi:softprob', 
        num_class=num_clases,
        eval_metric='mlogloss'
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200,          # Aumentamos árboles para captar más detalle
        learning_rate=0.05,        # Aprendizaje más lento y preciso
        num_leaves=31,             # Complejidad del árbol estándar
        min_child_samples=5,       # CLAVE: Permite crear hojas con pocos datos (importante para celdas poco visitadas)
        objective='multiclass',    # Forzamos clasificación multiclase
        num_class=num_clases,      # Número total de celdas
        random_state=42,
        verbose=-1,
        force_col_wise=True
    )
}

# 8. Bucle de entrenamiento
resultados = []
for nombre, model in modelos.items():
    inicio = time.time()
    print(f"Entrenando {nombre}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultados.append({"Modelo": nombre, "Accuracy": acc, "Tiempo (s)": time.time() - inicio})
    print(f"✅ {nombre} finalizado. Accuracy: {acc:.4f}")

print("\n--- COMPARATIVA FINAL (POR SEGMENTOS) ---")
print(pd.DataFrame(resultados).sort_values(by="Accuracy", ascending=False))

Entrenando Random Forest...
✅ Random Forest finalizado. Accuracy: 0.8082
Entrenando XGBoost...
✅ XGBoost finalizado. Accuracy: 0.8013
Entrenando LightGBM...
✅ LightGBM finalizado. Accuracy: 0.3057

--- COMPARATIVA FINAL (POR SEGMENTOS) ---
          Modelo  Accuracy  Tiempo (s)
0  Random Forest  0.808234    5.203001
1        XGBoost  0.801328   34.477465
2       LightGBM  0.305710   71.558020


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# 1. Rendimiento: Entrenamiento vs Test (Detección de Overfitting)
# Usamos el modelo que ya tienes entrenado en el diccionario
modelo_rf = modelos["Random Forest"]
train_acc = modelo_rf.score(X_train, y_train)
y_pred_test = modelo_rf.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)

# 2. Complejidad del problema
num_filas_train = len(X_train)
num_celdas_unicas = len(le_final.classes_)

# 3. Análisis de desbalanceo (Celdas más frecuentes vs menos frecuentes)
counts = pd.Series(y_train).value_counts()
top_1_percent = int(max(1, len(counts) * 0.01))
cobertura_top = counts.iloc[:top_1_percent].sum() / len(y_train) * 100

# 4. Cálculo de Accuracy Top-3 (Probabilidad de estar en el "vecindario" correcto)
y_probs = modelo_rf.predict_proba(X_test)
top3_indices = np.argsort(y_probs, axis=1)[:, -3:]
y_test_array = np.array(y_test).reshape(-1, 1)
hits_top3 = np.any(top3_indices == y_test_array, axis=1)
top3_acc = np.mean(hits_top3)

# --- SALIDA DE DATOS ---
print("--- DIAGNÓSTICO FINAL DE RENDIMIENTO ---")
print(f"1. Accuracy en Entrenamiento: {train_acc:.4f}")
print(f"2. Accuracy en Test (Exacto): {test_acc:.4f}")
print(f"3. Accuracy Top-3 (Vecindario): {top3_acc:.4f}")
print(f"4. Diferencia (Gap):         {train_acc - test_acc:.4f}")
print("-" * 40)
print(f"5. Total filas en Train:      {num_filas_train}")
print(f"6. Total celdas a predecir:   {num_celdas_unicas}")
print(f"7. Ratio Datos/Celdas:        {num_filas_train / num_celdas_unicas:.2f}")
print("-" * 40)
print(f"8. Concentración de datos: El 1% de las celdas más frecuentes concentra el {cobertura_top:.2f}% de los datos.")

# Avisos metodológicos
if top3_acc >= 0.90:
    print("\n✅ ¡OBJETIVO CONSEGUIDO! El modelo sitúa al ave en el área correcta con un 90% de confianza (Top-3).")

if train_acc > 0.95 and (train_acc - test_acc) > 0.20:
    print("⚠️ AVISO: Hay indicios de OVERFITTING (el modelo memoriza demasiado el pasado).")

--- DIAGNÓSTICO FINAL DE RENDIMIENTO ---
1. Accuracy en Entrenamiento: 1.0000
2. Accuracy en Test (Exacto): 0.8082
3. Accuracy Top-3 (Vecindario): 0.8736
4. Diferencia (Gap):         0.1918
----------------------------------------
5. Total filas en Train:      16323
6. Total celdas a predecir:   952
7. Ratio Datos/Celdas:        17.15
----------------------------------------
8. Concentración de datos: El 1% de las celdas más frecuentes concentra el 42.63% de los datos.
